# Exploratory Data Analysis (EDA) - PGA Prediction
Notebook ini bertugas untuk mengeksplorasi dataset `indonesia_earthquake_cleaned.csv` guna keperluan Data Analysis & Understanding.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
%matplotlib inline

# Atur style visual
sns.set_theme(style="whitegrid")

## 1. Load Data & Inspection

In [ ]:
data_path = "../data/raw/indonesia_earthquake_cleaned.csv"
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
display(df.head())

In [ ]:
df.info()

## 2. Missing Value Diagnostics

In [ ]:
missing_vals = df.isnull().sum()
missing_percentages = (missing_vals / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_vals,
    'Missing_Percentage': missing_percentages
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Missing_Percentage', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Missing_Percentage', y=missing_df.index, data=missing_df)
plt.title("Persentase Missing Values per Kolom")
plt.xlabel("Percentage (%)")
plt.ylabel("Features")
plt.show()

## 3. Univariate Analysis
Target variabel utama kita: `maxpga`. Fitur prediktor kunci utamanya adalah `mag`, `depth_e`. 
Catatan: Fitur rekam kejadian seperti `maxmmi`, `maxpsa03`, `maxpsa10`, `maxpsa30` adalah *secondary targets* yang sebaiknya **dihapus (dropped)** dalam pemodelan prediktor agar tidak membocorkan informasi (*data leakage*).

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df['maxpga'], kde=True, bins=50, color='firebrick')
plt.title("Distribusi Peak Ground Acceleration (PGA)")

plt.subplot(1, 2, 2)
sns.boxplot(x=df['maxpga'], color='salmon')
plt.title("Boxplot PGA")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(df['mag'], kde=True, bins=30, ax=axes[0], color='teal')
axes[0].set_title("Distribusi Magnitude (Mw)")

sns.histplot(df['depth_e'], kde=True, bins=30, ax=axes[1], color='navy')
axes[1].set_title("Distribusi Kedalaman (km)")

plt.tight_layout()
plt.show()

## 4. Bivariate Analysis & Correlation
Hubungan kedalaman dan besaran magnetudo terhadap maxpga.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x='mag', y='maxpga', alpha=0.5, ax=axes[0], color='darkorange')
axes[0].set_title("Magnitude vs PGA")

sns.scatterplot(data=df, x='depth_e', y='maxpga', alpha=0.5, ax=axes[1], color='darkcyan')
axes[1].set_title("Depth vs PGA")

plt.tight_layout()
plt.show()

In [ ]:
# Memilih fitur utama untuk menghitung matriks korelasi
features_to_check = ['maxpga', 'mag', 'depth_e', 'latitude_e', 'longitude_e', 'dmin', 'rms', 'gap']
corr_matrix = df[features_to_check].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of Key Features")
plt.show()

## 5. Spatial Diagnostics (Distribusi Spasial Geografis)
Memetakan koordinat (`latitude_e`, `longitude_e`) dan melihat secara visual tingkat sebaran `maxpga`.

In [ ]:
plt.figure(figsize=(12, 6))
scatter = plt.scatter(x=df['longitude_e'], y=df['latitude_e'], 
                      c=df['maxpga'], cmap='Reds', alpha=0.7, s=df['mag']*2)
plt.colorbar(scatter, label='Peak Ground Acceleration (maxpga)')
plt.title("Peta Persebaran Gempa berdasarkan maxpga di Indonesia")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()